# EO/IR Füzyon Projesi: ham tespitler, ölçüm sağlaması ve füzyon ayarı

Bu defter eğitim defterinden sonra çalışır. Sıra:

1. Her iki `best.pt` ile val ve test için **ham tespitleri** kaydet (`conf=0.001`). Metrik hesaplanmaz, test'e bakılmaz.
2. **Sağlama:** kendi değerlendiricimiz (`evaluate_dets.py`) val'de Ultralytics'in kendi `val()` sonucunu birkaç binde bir içinde tutturmalı. Tutmazsa füzyona geçme.
3. **Füzyonu val'de ayarla:** klip bazlı çapraz doğrulama (`tune_fusion.py`), seçilen ayar `fusion_config.json` olarak dondurulur.
4. **Test:** yalnızca 3. adım bitip kararlar dondurulunca, tek sefer. Hücre yorum satırı halinde.

**Önce:** Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU. 1. adım GPU ister, 2 ve 3 CPU'da da çalışır.

## 0. Kurulum

In [ ]:
!pip -q install ultralytics==8.4.131 ensemble-boxes==1.0.9
import torch, ultralytics
print("CUDA:", torch.cuda.is_available(), "| ultralytics", ultralytics.__version__)

Repo kodunu al. Repo gizliyse `git clone` yerine repoyu zip olarak Drive'a koyup `/content/eo-ir-fusion` altına aç.

In [ ]:
BRANCH = "main"   # kodun bulundugu dal
!rm -rf /content/eo-ir-fusion && git clone -q -b {BRANCH} https://github.com/ebrudeniz/eo-ir-fusion.git /content/eo-ir-fusion
%cd /content/eo-ir-fusion
!git log --oneline -1

## 1. Drive ve veri

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_DIR = Path("/content/drive/MyDrive/eo_ir_fusion")
WEIGHTS = {m: DRIVE_DIR / "runs" / f"{m}_yolov8n_e50" / "weights" / "best.pt" for m in ("ir", "eo")}
DETS = DRIVE_DIR / "dets"            # ham tespitler buraya (Drive'da kalsin)
DETS.mkdir(parents=True, exist_ok=True)
for m, w in WEIGHTS.items():
    assert w.exists(), f"Agirlik yok: {w}"
print(WEIGHTS)

In [ ]:
for m in ("ir", "eo"):
    if not Path(f"/content/dataset_{m}").exists():
        !cp "{DRIVE_DIR}/dataset_{m}.zip" /content/
        !unzip -q -o "/content/dataset_{m}.zip" -d /content
    for split, n in {"val": 2104, "test": 3463}.items():
        k = len(list(Path(f"/content/dataset_{m}/images/{split}").glob("*.jpg")))
        assert k == n, (m, split, k)
print("Tamam: val 2104, test 3463, iki modalitede")

## 2. Ham tespitler (GPU)
`predict_raw.py` girdiyi Ultralytics val ile aynı boyuta getiriyor (1280x1024 için 544x640, ortalanmış dolgu). Test tespitleri burada **üretilir ama ölçülmez**. Zaten varsa atlanır, yeniden üretmek için dosyayı sil.

In [ ]:
for m in ("ir", "eo"):
    for split in ("val", "test"):
        out = DETS / f"{m}_{split}.json.gz"
        if out.exists():
            print("var, atlandi:", out); continue
        !python3 scripts/predict_raw.py --weights "{WEIGHTS[m]}" --dataset_dir /content/dataset_{m} \
            --split {split} --modality {m} --out "{out}"

## 3. Sağlama: değerlendirici Ultralytics ile aynı sayıyı veriyor mu?
Referans, Ultralytics'in **aynı ayarlarla** (fp32, conf 0,001, IoU 0,7) val'de verdiği sayı. Eğitim sonundaki 0,958 / 0,616 ve 0,877 / 0,492 fp16 ile hesaplandığı için burada referansı yeniden alıyoruz. İki tablo da yanyana yazdırılır.

In [ ]:
import yaml
from ultralytics import YOLO

ref = {}
for m in ("ir", "eo"):
    cfg = {"path": f"/content/dataset_{m}", "train": "images/train", "val": "images/val",
           "test": "images/test", "names": {0: "person"}}
    yp = f"/content/dataset_{m}.yaml"
    yaml.safe_dump(cfg, open(yp, "w"), sort_keys=False)
    r = YOLO(str(WEIGHTS[m])).val(data=yp, split="val", imgsz=640, batch=16, conf=0.001, iou=0.7,
                                 half=False, plots=False, verbose=False)
    ref[m] = (round(r.box.map50, 4), round(r.box.map, 4))
print("Ultralytics referansi (fp32):", ref)

In [ ]:
EXPECT = " ".join(f"{m}={a},{b}" for m, (a, b) in ref.items())
!python3 scripts/evaluate_dets.py --dataset_dir /content/dataset_ir --split val \
    --dets ir="{DETS}/ir_val.json.gz" eo="{DETS}/eo_val.json.gz" \
    --expect {EXPECT} --tol 0.003 --per_clip --out "{DRIVE_DIR}/fusion_val/baselines_val.json"

Son satır `TAMAM` demeli. `TUTMUYOR` derse füzyona geçme: fark ön işlemeden mi (predict ile val girdisi), eşleştirmeden mi geliyor, önce onu bul.

## 4. Füzyonu val'de ayarla (CPU yeter)
4 val klibiyle bir-klip-dışarıda çapraz doğrulama, 45 füzyon ayarı ve WBF referansı. Birincil ölçüt önceden seçildi: out-of-fold mAP50-95.

In [ ]:
!python3 scripts/tune_fusion.py --dataset_dir /content/dataset_ir \
    --eo "{DETS}/eo_val.json.gz" --ir "{DETS}/ir_val.json.gz" --out_dir "{DRIVE_DIR}/fusion_val"

In [ ]:
import pandas as pd
df = pd.read_csv(DRIVE_DIR / "fusion_val" / "cv_results.csv")
df[["method", "map50", "map50_95", "precision", "recall"]].head(15)

`fusion_val/` klasörünü (`fusion_config.json`, `wbf_config.json`, `cv_results.csv`, `summary.json`, `reliability.png`, `baselines_val.json`) repoya `results/fusion_val/` olarak ekle. Ham tespit dosyaları büyük, repoya girmez.

## 5. Test (şimdi çalıştırma)
Yalnızca yöntem tamamen dondurulduktan sonra, **tek sefer**. `evaluate_dets.py --final` her ölçümü `results/test_eval_log.jsonl`'e yazar ve aynı dosyaların ikinci kez ölçülmesini engeller. Bu kayıt dosyasını da repoya ekle.

In [ ]:
# Hazir oldugunda yorumlari kaldir, TEK SEFER calistir:
# FV = DRIVE_DIR / "fusion_val"
# !python3 scripts/apply_fusion.py --config "{FV}/fusion_config.json" --eo "{DETS}/eo_test.json.gz" --ir "{DETS}/ir_test.json.gz" --out "{DETS}/fusion_test.json.gz"
# !python3 scripts/apply_fusion.py --config "{FV}/wbf_config.json" --eo "{DETS}/eo_test.json.gz" --ir "{DETS}/ir_test.json.gz" --out "{DETS}/wbf_test.json.gz"
# !python3 scripts/evaluate_dets.py --dataset_dir /content/dataset_ir --split test --final \
#     --dets ir="{DETS}/ir_test.json.gz" eo="{DETS}/eo_test.json.gz" fusion="{DETS}/fusion_test.json.gz" wbf="{DETS}/wbf_test.json.gz" \
#     --per_clip --bootstrap 1000 --reference ir --note "Faz 1 nihai test" --out "{DRIVE_DIR}/test_results.json"
# !cp results/test_eval_log.jsonl "{DRIVE_DIR}/"